## Setup

## Connect GDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### Dependency installs

In [ ]:
!pip install plyfile

### Colab environment (known-good)

This notebook compiles custom CUDA extensions (`diff_gaussian_rasterization`,
`simple-knn`) from source, pinned to commits built for a specific CUDA toolkit.
Newer Colab images can break that build — pip only reports the unhelpful
*"No available output."*

- **Known-good Colab runtime: `2026.10`.** If the extension build fails, set it in
  `Runtime → Change runtime type → Runtime version` and re-run from the top.
- The rasterizer needs `nvcc`'s CUDA to match `torch.version.cuda`; the cell below
  checks this and fails fast with the real reason.
- Order of debugging: switch the runtime version **first**; only then edit the
  rasterizer source.


### Check GPU and CUDA availability

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
from torch.utils.tensorboard import SummaryWriter
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# --- Toolchain consistency check -------------------------------------------------
# diff_gaussian_rasterization is pinned to a commit built for a specific CUDA
# toolkit.  torch.utils.cpp_extension aborts the wheel build when nvcc's CUDA
# does not match torch.version.cuda -- and pip reports that only as the unhelpful
# "No available output".  This cell fails fast with the real reason instead.
#
# Known-good Colab runtime: 2026.10 (CUDA 12.x toolchain).
import re, shutil, subprocess, warnings

PINNED_RASTERIZER_CUDA_MAJOR = "12"     # the pinned rasterizer is fixed for CUDA 12.x


def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    try:
        out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    except Exception:
        return None
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None


torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print("\nCUDA toolchain check:")
print(f"  torch        : {torch.__version__} (built for CUDA {torch_cuda})")
print(f"  nvcc         : {nvcc_cuda or 'NOT FOUND on PATH'}")

if nvcc_cuda is None:
    warnings.warn("nvcc not found on PATH -- the CUDA extensions cannot be built. "
                  "Select a Colab runtime that ships nvcc (known-good: 2026.10).")
elif torch_cuda is None:
    warnings.warn("This PyTorch build has no CUDA; the extensions cannot be built.")
elif torch_cuda.split(".")[0] != nvcc_cuda.split(".")[0]:
    raise RuntimeError(
        f"CUDA major mismatch: torch was built for CUDA {torch_cuda} but nvcc is {nvcc_cuda}. "
        "Building diff_gaussian_rasterization will fail. Select a Colab runtime with a matching "
        "toolchain (known-good: 2026.10) or install a matching torch.")
elif torch_cuda.split(".")[0] != PINNED_RASTERIZER_CUDA_MAJOR:
    warnings.warn(
        f"torch/nvcc are CUDA {torch_cuda}, but the pinned rasterizer targets CUDA "
        f"{PINNED_RASTERIZER_CUDA_MAJOR}.x. The wheel build may fail with 'No available output'. "
        "Known-good runtime: 2026.10.")
else:
    print("  OK: toolchain is consistent with the pinned rasterizer.")


### Clone 3DGS repo & install submodules

In [ ]:
# %%bash
# echo "Cloning 3D Gaussian Splatting repo..."
# if [ ! -d "gaussian_splatting" ]; then
#   git clone --recursive https://github.com/satyaba/gaussian-splatting.git gaussian_splatting
# fi

%cd /content
!rm -rf gaussian-splatting
!git clone -q https://github.com/satyaba/gaussian-splatting.git
%cd gaussian-splatting

!git submodule update --init --recursive
!test -f submodules/diff-gaussian-rasterization/third_party/glm/glm/glm.hpp && echo 'glm OK' || echo 'glm MISSING'
# !git submodule update --init submodules/diff-gaussian-rasterization submodules/simple-knn submodules/fused-ssim

# MUST print 813db1d (the kernel seg vote path). If not, the clone is stale - STOP.
!git -C submodules/diff-gaussian-rasterization log --oneline -1

In [ ]:
%%bash
cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
pip install . --no-build-isolation 2>&1 | tail -60

cd /content/gaussian-splatting
pip install kmeans-pytorch

In [ ]:
%%bash
cd /content/gaussian-splatting
pip install ./submodules/simple-knn --no-build-isolation -q 2>&1 | tail -2
python -c "import torch; from simple_knn._C import distCUDA2; print('simple_knn OK')"

In [ ]:
%%bash
cd /content
python -c "import diff_gaussian_rasterization; print('installed at:', diff_gaussian_rasterization.__file__)"

### Import modules

In [ ]:
import sys
import seaborn as sns

sys.path.append('/content/gaussian-splatting')

In [ ]:
from pathlib import Path
import numpy as np
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from random import randint
from gaussian_renderer import render
from scipy.spatial import cKDTree

import argparse
import csv
import glob
import json
import os
import re
import cv2

In [ ]:
RUN_ID = 'run40'

HOME_DIR = Path('/content/drive/MyDrive/Magister/Thesis/Script')
DATASET_DIR = HOME_DIR / 'Dataset' / 'office_0' / 'Sequence_1'
SCENE_DIR =  DATASET_DIR / 'images'
SEG_DIR = DATASET_DIR / 'semantic_class'
OUT_DIR = HOME_DIR / 'output' / RUN_ID

NUM_CLASSES = None

In [ ]:
config = {
    'LVL1_FLOATERS': True,
    'LVL2_FLOATERS': True,
    # Option A: the clone/split trigger accumulates only the photometric
    # screen-space gradient, so the segmentation loss cannot inflate it.
    'DENSIFY_PHOTO_GRAD_ONLY': True,
    # 'POSITION_BASED_THRESH': {
    #     'boundary_weight': 0.5,
    #     'interior_weight': 1.0,
    #     'other_weight': 1.0,
    # },
    'POSITION_BASED_THRESH': False,
    # Floater lifecycle: 'prune' (run26: remove every event -> churn) or
    # 'freeze' (keep, de-candidate for clone/split -> no churn).
    'FLOATER_ACTION': 'freeze',
    # 'TEXTURE_BASED_THRESH': {
    #     'solver': 'laplacian_energy',
    #     'beta': 0.5
    # }
    'TEXTURE_BASED_THRESH': False,
    'TEXTURE_BASED_SCALE': {
        'solver': 'laplacian_energy',
        'beta': 0.5
    }
}

### Inspect with Tensorboard

In [ ]:
!pip install tensorboard
%load_ext tensorboard
%tensorboard --logdir ./logs

## Segmentation Audit

In [ ]:
# Audit cell: total classes available in segmentation files (run BEFORE Cell 4)

import glob, os
import numpy as np
import cv2

assert os.path.isdir(SEG_DIR), f"segmentation folder not found: {SEG_DIR}"

# mirror dataset_readers._segmentation_path_for (scene/dataset_readers.py:72-82):
# <seg_dir>/<split>/<stem>.png preferred, <seg_dir>/<stem>.png fallback.
# Exclude vis_* files — Replica semantic_class/ ships one visualization render
# per GT map (verified locally: 900 GT + 900 vis RGB PNGs per sequence); vis
# files are colored images that would poison the class-id histogram.
paths = sorted({*glob.glob(os.path.join(SEG_DIR, "train", "*.png")),
                *glob.glob(os.path.join(SEG_DIR, "test",  "*.png")),
                *glob.glob(os.path.join(SEG_DIR, "*.png"))})
paths = [p for p in paths if "vis_" not in os.path.basename(p)]
assert paths, f"no .png under {SEG_DIR}"

# ---- accumulate a class-id histogram (uint8 domain) ---------------------
counts = np.zeros(256, dtype=np.int64)      # index = class id
void_fracs, dtypes = [], set()

for p in tqdm(paths, desc="scanning seg maps"):
    raw = cv2.imread(p, -1)                  # same call as camera_utils.py:68
    assert raw is not None, p
    if raw.ndim == 3:                        # same channel-0 rule as :70-71
        raw = raw[..., 0]
    dtypes.add(str(raw.dtype))
    h = np.bincount(raw.ravel(), minlength=256)
    counts += h
    void_fracs.append(h[255] / h.sum())

# ---- report -------------------------------------------------------------
valid   = counts.copy(); valid[255] = 0      # 255 = void, not a class
present = np.nonzero(valid)[0]
total   = counts.sum()

print(f"\nfolder          : {SEG_DIR}")
print(f"files           : {len(paths)}  (dtypes seen: {', '.join(sorted(dtypes))})")
print(f"distinct classes: {len(present)}")
print(f"max class id    : {present.max()}")
print(f"void (255)      : {counts[255] / total:.2%} of all pixels "
      f"(per-frame min/max: {min(void_fracs):.2%} / {max(void_fracs):.2%})")
print(f"\nid  pixels        share")
for cid in present:
    print(f"{cid:3d} {valid[cid]:>13,d}  {valid[cid] / valid.sum():6.2%}")

# ---- gate, mirroring camera_utils.py:78-80 ------------------------------
# if present.max() >= dataset.num_segmentation_classes:
#     raise AssertionError(
#         f"taxonomy mismatch: max id {present.max()} >= "
#         f"num_segmentation_classes={dataset.num_segmentation_classes} "
#         f"(loader would assert at load time)")
# print(f"\nOK: {len(present)} classes fit in num_segmentation_classes={dataset.num_segmentation_classes}")
print(f"suggested num_segmentation_classes: {present.max() + 1}  "
      f"(= max_id + 1, NOT the distinct count {len(present)})")

NUM_CLASSES = present.max() + 1

rare = present[valid[present] < 0.0001 * valid.sum()]   # <0.01% of labeled px
if len(rare):
    print(f"rare classes (<0.01% of labeled px): {rare.tolist()}")

## Class Texture Computation

In [ ]:
IGNORE = []
ERODE = 3
MIN_PX = 500
MIN_VIEWS = 3
K = None
STEP = 1

NO_ILLUM_NORM = True
BLUR_SIGMA = 1.0

NAMES = None

VIZ = 5

In [ ]:
# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def last_int(path):
    """Last integer in the file stem, used to pair rgb <-> seg files."""
    nums = re.findall(r"\d+", os.path.splitext(os.path.basename(path))[0])
    return int(nums[-1]) if nums else None


def pair_files(rgb_files, seg_files):
    rgb_map = {last_int(f): f for f in rgb_files if last_int(f) is not None}
    seg_map = {last_int(f): f for f in seg_files if last_int(f) is not None}
    ids = sorted(set(rgb_map) & set(seg_map))
    skipped = (len(rgb_map) - len(ids), len(seg_map) - len(ids))
    if skipped != (0, 0):
        print(f"[warn] unmatched files skipped: rgb={skipped[0]}, seg={skipped[1]}")
    if not ids:
        raise SystemExit("No rgb/seg pairs matched by numeric id in the filename.")
    return [(i, rgb_map[i], seg_map[i]) for i in ids]


def load_seg(path, hw):
    seg = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if seg is None:
        raise IOError(f"cannot read {path}")
    if seg.ndim == 3:
        seg = seg[..., 0]
    seg = seg.astype(np.int32)
    if seg.shape != hw:  # labels must never be interpolated
        seg = cv2.resize(seg, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    return seg


def laplacian_energy(rgb_bgr, illum_norm=True, blur_sigma=1.0, illum_sigma=10.0):
    """Per-pixel squared Laplacian response of the (normalised) grayscale image."""
    g = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    if illum_norm:  # remove slow brightness variation so bright != detailed
        local_mean = cv2.GaussianBlur(g, (0, 0), illum_sigma)
        g = g / (local_mean + 0.05)
    if blur_sigma > 0:  # suppress sensor / compression noise
        g = cv2.GaussianBlur(g, (0, 0), blur_sigma)
    lap = cv2.Laplacian(g, cv2.CV_32F, ksize=3)
    return lap * lap


def percentile_rank(x):
    """Rank of each value scaled to [0, 1] (smallest = 0, largest = 1)."""
    n = len(x)
    if n == 1:
        return np.array([0.5], dtype=np.float32)
    ranks = np.empty(n, dtype=np.float64)
    ranks[np.argsort(x, kind="stable")] = np.arange(n)
    return (ranks / (n - 1)).astype(np.float32)


In [ ]:
def compute_texture_freq():
  pairs = pair_files(sorted(glob.glob(str(SCENE_DIR / "*.png"))), sorted(glob.glob(str(SEG_DIR / "*.png"))))
  pairs = pairs[:: STEP]

  C = int(NUM_CLASSES)
  ksz = 2 * ERODE + 1
  kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz)) if ERODE > 0 else None

  sum_e = np.zeros(C, dtype=np.float64)   # sum of squared Laplacian over eroded masks
  cnt = np.zeros(C, dtype=np.int64)       # eroded pixel count
  views = np.zeros(C, dtype=np.int32)

  # ---- accumulate over views --------------------------------------------
  for n, (vid, rgb_path, seg_path) in enumerate(pairs):
    img = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
    if img is None:
        print(f"[warn] cannot read {rgb_path}, skipping")
        continue
    seg = load_seg(seg_path, img.shape[:2])
    energy = laplacian_energy(img, not NO_ILLUM_NORM, BLUR_SIGMA)
    # print(energy)

    for c in np.unique(seg):
        if c < 0 or c >= C or c in IGNORE:
            continue
        m = (seg == c).astype(np.uint8)
        if kernel is not None:
            m = cv2.erode(m, kernel)
        k = int(m.sum())
        if k < MIN_PX:
            continue
        sel = m.astype(bool)
        sum_e[c] += float(energy[sel].sum())
        cnt[c] += k
        views[c] += 1
    if (n + 1) % 20 == 0:
        print(f"  processed {n + 1}/{len(pairs)}")

  valid = (views >= MIN_VIEWS) & (cnt > 0)
  if valid.sum() == 0:
      raise SystemExit("No class passed the min-views / min-px filters.")

  raw = np.full(C, np.nan)
  raw[valid] = sum_e[valid] / cnt[valid]
  log_raw = np.full(C, np.nan)
  log_raw[valid] = np.log(raw[valid] + 1e-12)

  prior = np.median(log_raw[valid])
  k = K if K is not None else float(np.percentile(cnt[valid], 25))
  w = np.zeros(C)
  w[valid] = cnt[valid] / (cnt[valid] + k)
  log_shrunk = np.full(C, np.nan)
  log_shrunk[valid] = w[valid] * log_raw[valid] + (1 - w[valid]) * prior

  s_tilde = np.full(C, 0.5, dtype=np.float32)  # unseen / ignored -> neutral
  s_tilde[valid] = percentile_rank(log_shrunk[valid])

  return s_tilde, k, float(prior), len(pairs)

In [ ]:
NUM_CLASSES

In [ ]:
if config['TEXTURE_BASED_THRESH'] or config['TEXTURE_BASED_SCALE']:
  s_tilde, k, prior, pair_count = compute_texture_freq()

Save computed texture frequency for possible future uses

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
if config['TEXTURE_BASED_THRESH']:
  np.save(OUT_DIR  / "s_tilde.npy", s_tilde)
  with open(os.path.join(OUT_DIR, "s_tilde.json"), "w") as f:
      json.dump({"s_tilde": s_tilde.tolist(), "shrinkage_k_px": k,
                  "prior_log_median": prior, "erode_px": ERODE,
                  "num_views": pair_count}, f, indent=1)

## 3DGS Init

In [ ]:
%cd /content/gaussian-splatting

import os, random, uuid
import numpy as np
import torch
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
from utils.general_utils import get_expon_lr_func
from utils.loss_utils import l1_loss, ssim
from utils.image_utils import psnr
from torch.utils.tensorboard import SummaryWriter

# seed parity with train.py's safe_state (utils/general_utils.py:130-133)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
torch.cuda.set_device(torch.device("cuda:0"))
assert torch.cuda.is_available()

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except ImportError:
    FUSED_SSIM_AVAILABLE = False
try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except ImportError:
    SPARSE_ADAM_AVAILABLE = False

def param_group(cls, **overrides):
    """Build a <X>Params GroupParams without argparse: defaults come from the
    class (stays in sync if you add attributes), per-run overrides via kwargs.
    Unlike the CLI, bools here CAN be set False explicitly."""
    pg = cls(ArgumentParser())          # registers defaults onto the instance
    d = {k.lstrip("_"): v for k, v in vars(pg).items()}
    d.update(overrides)
    return pg.extract(Namespace(**d))

In [ ]:
dataset = param_group(ModelParams,
    source_path=DATASET_DIR,
    model_path=f"{OUT_DIR}",
    resolution=1, eval=True,
    seg_encoding_dim=32,                      # == NUM_SEG_CHANNELS in cuda_rasterizer/config.h
    num_segmentation_classes=NUM_CLASSES,
    segmentation_path=SEG_DIR,         # "" disables L_seg
)
opt  = param_group(OptimizationParams, iterations=12_000)
pipe = param_group(PipelineParams)

assert dataset.seg_encoding_dim == 32          # same assert as train.py:60

print(dataset.sh_degree, opt.lambda_seg, pipe.debug)

In [ ]:
if not dataset.model_path:
    dataset.model_path = os.path.join("./output/", str(uuid.uuid4())[:10])
os.makedirs(dataset.model_path, exist_ok=True)
print("Output folder:", dataset.model_path)
# same file train.py:254 writes — downstream scripts merge it via get_combined_args
with open(os.path.join(dataset.model_path, "cfg_args"), "w") as f:
    f.write(str(Namespace(**vars(dataset))))

In [ ]:
def seg_hist_readout(seg_hist, eps=1e-8):
    """Read-outs from the persisted mass histogram M (doc 03 §3.2).

    Returns (consistency, structure, disagree, S), each [N]:
      consistency = max_c P[g, c]     (replaces the old top1/tot proxy)
      structure   = P_(1) + P_(2)     (boundary-ness)
      disagree    = 1 - P_(1)
      S           = sum_c M[g, c]     (evidence = mass, NOT count)
    Rows with S == 0 give P == 0, so their 'disagree' looks maximal — gate
    them out with S (`S > seg_hist_min_evidence`), never with disagree.
    """
    S = seg_hist.sum(axis=1)
    P = seg_hist / S.clamp_min(eps).unsqueeze(1)

    p = P + 1e-10
    entropy = -(p * torch.log2(p)).sum(dim=-1)
    k = min(2, P.shape[1])
    top2 = torch.topk(P, k=k, dim=1).values
    consistency = top2[:, 0]
    structure = top2[:, 0] + top2[:, 1] if k == 2 else top2[:, 0]
    disagree = 1.0 - top2[:, 0]
    return consistency, structure, disagree, entropy, top2, S

## Train Pipeline

In [ ]:
from scene import Scene, GaussianModel

gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type,
                          seg_encoding_dim=dataset.seg_encoding_dim,
                          num_segmentation_classes=dataset.num_segmentation_classes)
scene = Scene(dataset, gaussians)

In [ ]:
gaussians.training_setup(opt)

In [ ]:
from scene.segmentation_decoder import SegmentationDecoder, save_decoder_checkpoint

decoder = SegmentationDecoder(dataset.seg_encoding_dim, dataset.num_segmentation_classes).cuda()
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=opt.decoder_lr_init)
decoder_lr_decay_args = get_expon_lr_func(opt.decoder_lr_init, opt.decoder_lr_final,
                                        max_steps=max(1, opt.decoder_lr_decay_iters))

def get_decoder_lr(it):
        # Flat until decay_start, then exponential — decay begins BEFORE
        # seg_warmup_iters ends so confidence stabilizes early.
        if it < opt.decoder_lr_decay_start:
            return opt.decoder_lr_init
        return decoder_lr_decay_args(it - opt.decoder_lr_decay_start)

bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

iter_start = torch.cuda.Event(enable_timing = True)
iter_end = torch.cuda.Event(enable_timing = True)

use_sparse_adam = opt.optimizer_type == "sparse_adam" and SPARSE_ADAM_AVAILABLE
depth_l1_weight = get_expon_lr_func(opt.depth_l1_weight_init, opt.depth_l1_weight_final, max_steps=opt.iterations)

viewpoint_stack = scene.getTrainCameras().copy()
viewpoint_indices = list(range(len(viewpoint_stack)))
ema_loss_for_log = 0.0
ema_Ll1depth_for_log = 0.0
ema_Ll1seg_for_log = 0.0

### Densification

In [ ]:
from utils.general_utils import inverse_sigmoid, get_expon_lr_func, build_rotation

In [ ]:
def class_freq_thres(old_thresh, s_tilde, beta):
    return np.exp(-beta * (s_tilde - 0.5)) * old_thresh

In [ ]:
def densify_and_prune(gaussians, max_grad, min_opacity, extent, max_screen_size, radii,
                          interior_mask=None, boundary_mask=None, floater_mask=None, other_mask=None):
    gaussians.tmp_radii = radii                       # set BEFORE prune_points (it prunes tmp_radii)

    # FLOATER_ACTION controls the lifecycle of flagged floaters:
    #   'prune'  -> remove them (run26); recurs every event and causes churn
    #   'freeze' -> keep them, de-candidate for clone/split (no churn)
    freeze_mask = None
    floater_action = config.get('FLOATER_ACTION', 'prune')
    if floater_mask is not None and floater_action == 'prune':
        tr.on_prune(floater_mask, "floater")
        gaussians.prune_points(floater_mask)
        keep = ~floater_mask
        interior_mask = interior_mask[keep] if interior_mask is not None else None
        boundary_mask = boundary_mask[keep] if boundary_mask is not None else None
        other_mask = other_mask[keep] if other_mask is not None else None
    elif floater_mask is not None and floater_action == 'freeze':
        freeze_mask = floater_mask

    grads = gaussians.xyz_gradient_accum / gaussians.denom   # recomputed: any prune above changed the length
    grads[grads.isnan()] = 0.0

    # FREEZE is enforced at the clone/split selection site below (freeze_mask
    # argument), not via a gradient sentinel, so it is robust to any threshold.

    if config['POSITION_BASED_THRESH']:
        # NB: floaters are not members of any regime (interior/boundary/other),
        # so under position-based routing they are already excluded from
        # clone/split -- freeze_mask is therefore redundant here and not passed.
        regimes = {                                  # name -> (mask, scaling_threshold)
            "interior": (interior_mask, config['POSITION_BASED_THRESH']['interior_weight']),
            "boundary": (boundary_mask, config['POSITION_BASED_THRESH']['boundary_weight']),
            "other":  (other_mask,  config['POSITION_BASED_THRESH']['other_weight']),
        }

        for name, (m, t) in regimes.items():
            if m is None:
                continue
            tr.regime = name
            m = _pad_mask(m, gaussians.get_xyz.shape[0])
            densify_and_clone(gaussians, grads, max_grad, extent,
                                  dense_weight=t, sel_mask=m)

        # 3. SPLITS second. Each split removes its parents, shifting indices, so
        #    every not-yet-run sibling mask is remapped onto the new ordering.
        names = list(regimes.keys())
        for i, name in enumerate(names):
            m, t = regimes[name]
            if m is None:
                continue
            tr.regime = name
            m = _pad_mask(m, gaussians.get_xyz.shape[0])
            prune_filter, n_children = densify_and_split(gaussians, grads, max_grad, extent,
                                            dense_weight=t, sel_mask=m)
            for rest in names[i + 1:]:
                rm, rt = regimes[rest]
                if rm is None:
                    continue
                rm = _pad_mask(rm, prune_filter.shape[0])
                regimes[rest] = (_remap_after_split(rm, prune_filter), rt)

    else:
        tr.regime = "all"
        densify_and_clone(gaussians, grads, max_grad, extent,
                                  dense_weight=1, sel_mask=None, freeze_mask=freeze_mask)
        prune_filter, n_children = densify_and_split(gaussians, grads, max_grad, extent,
                                            dense_weight=1, sel_mask=None, freeze_mask=freeze_mask)

    prune_mask = (gaussians.get_opacity < min_opacity).squeeze()
    prune_reason = "opacity"

    if max_screen_size:
        big_points_vs = gaussians.max_radii2D > max_screen_size
        big_points_ws = gaussians.get_scaling.max(dim=1).values > 0.1 * extent
        prune_mask = torch.logical_or(torch.logical_or(prune_mask, big_points_vs), big_points_ws)
        prune_reason = "opacity+size"
    tr.on_prune(prune_mask, prune_reason)
    gaussians.prune_points(prune_mask)
    gaussians.tmp_radii = None

    torch.cuda.empty_cache()

In [ ]:
def densify_and_clone(gaussians, grads, grad_threshold, scene_extent, dense_weight=1, sel_mask=None, freeze_mask=None):
    # Extract points that satisfy the gradient condition
    n_init_points = gaussians.get_xyz.shape[0]

    # sibling regime can never re-clone them (doc 02 §5).
    padded_grad = torch.zeros((n_init_points), device="cuda")
    padded_grad[:grads.shape[0]] = torch.norm(grads, dim=-1).squeeze()

    if config['TEXTURE_BASED_THRESH']:
        new_thresh = torch.tensor(class_freq_thres(grad_threshold, s_tilde, config['TEXTURE_BASED_THRESH']['beta']), device='cuda')
        gauss_class_ids = torch.argmax(gaussians.seg_hist, dim=1)
        gauss_thresh = new_thresh[gauss_class_ids]
    else:
        gauss_thresh = grad_threshold

    selected_pts_mask = padded_grad >= gauss_thresh

    basic_thresh = dense_weight * gaussians.percent_dense * scene_extent

    if config['TEXTURE_BASED_SCALE']:
        new_scale_thresh = torch.tensor(class_freq_thres(basic_thresh, s_tilde, config['TEXTURE_BASED_SCALE']['beta']), device='cuda')
        gauss_class_ids = torch.argmax(gaussians.seg_hist, dim=1)
        gauss_scale_thresh = new_scale_thresh[gauss_class_ids]
    else:
        gauss_scale_thresh = basic_thresh

    selected_pts_mask = torch.logical_and(selected_pts_mask,
                                          torch.max(gaussians.get_scaling, dim=1).values <= gauss_scale_thresh)
    if sel_mask is not None:                                                 # NEW (T1): interior regime
        selected_pts_mask = torch.logical_and(selected_pts_mask, sel_mask)
    if freeze_mask is not None:                                              # FREEZE: never clone
        selected_pts_mask = torch.logical_and(
            selected_pts_mask, ~_pad_mask(freeze_mask, n_init_points))

    new_xyz = gaussians._xyz[selected_pts_mask]
    new_features_dc = gaussians._features_dc[selected_pts_mask]
    new_features_rest = gaussians._features_rest[selected_pts_mask]
    new_seg_encoding = gaussians._segmentation_encoding[selected_pts_mask]
    new_seg_hist = gaussians.seg_hist[selected_pts_mask] * gaussians.seg_hist_inherit_discount
    new_opacities = gaussians._opacity[selected_pts_mask]
    new_scaling = gaussians._scaling[selected_pts_mask]
    new_rotation = gaussians._rotation[selected_pts_mask]

    new_tmp_radii = gaussians.tmp_radii[selected_pts_mask]

    gaussians.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling, new_rotation, new_seg_encoding, new_tmp_radii, new_seg_hist)

    tr.on_clone(selected_pts_mask, padded_grad, new_xyz.detach(), N=1)


In [ ]:
def densify_and_split(gaussians, grads, grad_threshold, scene_extent, N=2, dense_weight=1, sel_mask=None, freeze_mask=None):
    n_init_points = gaussians.get_xyz.shape[0]
    # Extract points that satisfy the gradient condition
    padded_grad = torch.zeros((n_init_points), device="cuda")
    padded_grad[:grads.shape[0]] = grads.squeeze()

    if config['TEXTURE_BASED_THRESH']:
        new_thresh = torch.tensor(class_freq_thres(grad_threshold, s_tilde, config['TEXTURE_BASED_THRESH']['beta']), device='cuda')
        gauss_class_ids = torch.argmax(gaussians.seg_hist, dim=1)
        gauss_thresh = new_thresh[gauss_class_ids]
    else:
        gauss_thresh = grad_threshold

    selected_pts_mask = torch.where(padded_grad >= gauss_thresh, True, False)

    basic_thresh = dense_weight * gaussians.percent_dense * scene_extent

    if config['TEXTURE_BASED_SCALE']:
        new_scale_thresh = torch.tensor(class_freq_thres(basic_thresh, s_tilde, config['TEXTURE_BASED_SCALE']['beta']), device='cuda')
        gauss_class_ids = torch.argmax(gaussians.seg_hist, dim=1)
        gauss_scale_thresh = new_scale_thresh[gauss_class_ids]
    else:
        gauss_scale_thresh = basic_thresh

    selected_pts_mask = torch.logical_and(selected_pts_mask,
                                          torch.max(gaussians.get_scaling, dim=1).values > gauss_scale_thresh)

    if sel_mask is not None:                                                 # NEW (T1): boundary regime
        selected_pts_mask = torch.logical_and(
            selected_pts_mask, _pad_mask(sel_mask, n_init_points))
    if freeze_mask is not None:                                              # FREEZE: never split
        selected_pts_mask = torch.logical_and(
            selected_pts_mask, ~_pad_mask(freeze_mask, n_init_points))

    # Vanilla split placement: children sampled from N(0, Sigma) in the
    # parent's local frame (T0 deterministic placement was removed).
    stds = gaussians.get_scaling[selected_pts_mask].repeat(N,1)
    means =torch.zeros((stds.size(0), 3),device="cuda")
    samples = torch.normal(mean=means, std=stds)

    rots = build_rotation(gaussians._rotation[selected_pts_mask]).repeat(N,1,1)
    new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1) + gaussians.get_xyz[selected_pts_mask].repeat(N, 1)
    new_scaling = gaussians.scaling_inverse_activation(gaussians.get_scaling[selected_pts_mask].repeat(N,1) / (0.8*N))
    new_rotation = gaussians._rotation[selected_pts_mask].repeat(N,1)
    new_features_dc = gaussians._features_dc[selected_pts_mask].repeat(N,1,1)
    new_features_rest = gaussians._features_rest[selected_pts_mask].repeat(N,1,1)
    new_seg_encoding = gaussians._segmentation_encoding[selected_pts_mask].repeat(N,1)
    new_seg_hist = gaussians.seg_hist[selected_pts_mask].repeat(N,1) * gaussians.seg_hist_inherit_discount
    new_opacity = gaussians._opacity[selected_pts_mask].repeat(N,1)
    new_tmp_radii = gaussians.tmp_radii[selected_pts_mask].repeat(N)

    gaussians.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacity, new_scaling, new_rotation, new_seg_encoding, new_tmp_radii, new_seg_hist)

    tr.on_split(selected_pts_mask, padded_grad, new_xyz.detach(), N=N)

    prune_filter = torch.cat((selected_pts_mask, torch.zeros(N * selected_pts_mask.sum(), device="cuda", dtype=bool)))
    n_children = int(new_xyz.shape[0])

    gaussians.prune_points(prune_filter)

    return prune_filter, n_children

In [ ]:
@staticmethod
def _pad_mask(mask, n):
    """Regime masks are length P_old. split runs AFTER clone (which appended
    rows at the tail), so a mask handed to split must be padded with False to
    the new length — a freshly created child is never pre-selected (doc 02 §5).
    None passes through unchanged (vanilla fallback)."""
    if mask is None or mask.shape[0] == n:
        return mask

    if mask.shape[0] > n:
        raise RuntimeError(
            f"_pad_mask: mask len {mask.shape[0]} > population {n}; "
            f"stale/over-remapped mask — layout bug, not a pad case")
    out = torch.zeros(n, dtype=torch.bool, device="cuda")
    out[:mask.shape[0]] = mask
    return out

In [ ]:
@staticmethod
def _remap_after_split(mask, prune_filter):
    """Prune keeps the appended children (their slots are False), so ~prune_filter
    is already the full new ordering (survivors + children). No re-append."""
    if mask is None:
        return None
    return mask[~prune_filter]

In [ ]:
"""
Grid-based geometric support test for Gaussians (reach / height).

GOAL
    For each query Gaussian (e.g. class-1), measure how well trusted reference
    Gaussians (e.g. class-2 "interior") support it:
      reach  = how far the nearest trusted neighbours are (sideways)
      height = how far the query sits off the plane those neighbours form
    Both are divided by the local spacing of the trusted Gaussians, so they are
    scale-free ("1.0" ~ one normal neighbour gap, anywhere in the scene).

DATA FLOW
    ref (trusted points) --build_grid--> grid g
    g + ref  --grid_knn(self_exclude)--> spacing field sp  (typical gap around each ref point)
    g + qry  --grid_knn-->               K nearest ref points per query (within radius h)
    neighbours --> reach, plane fit (PCA) --> normal --> height

SHAPE GLOSSARY
    N   number of reference points          Q   number of query points
    K   neighbours kept per query           M   max points read from ONE cell
    h   cell side length = search radius    27  the 3x3x3 block of cells around a query
"""

# The 27 integer offsets (-1/0/+1 on each axis) that enumerate a cell and its 26 neighbours.
OFFS = torch.stack(torch.meshgrid(*[torch.arange(-1, 2)] * 3, indexing='ij'), -1).view(-1, 3)  # [27,3]

def build_grid(pts, h):
    """
    Bucket points into cubic cells of side h and index them so that all points of one
    cell are stored contiguously. Think "pigeonholes for mail".

    pts: [N,3] float tensor.  Returns a dict (the "address book" of the grid).
    """
    mn = pts.min(0).values                                   # min corner of the scene = grid origin

    # Integer cell coordinate of every point: floor((x - origin) / h), per axis.
    # "+ 1" shifts all coords to start at 1, so the neighbour cell at offset -1 is still >= 0.
    ijk = ((pts - mn) / h).floor().long() + 1                # [N,3]

    # Grid extent per axis. "+ 2" leaves one empty layer on each side for the +/-1 neighbour lookup.
    dims = ijk.max(0).values + 2                             # [3]
    assert dims.prod().item() < 2**62, "grid too large; increase h or clip outliers"

    # Flatten (i,j,k) into ONE integer per point (row-major index). Same cell => same key.
    key = (ijk[:, 0] * dims[1] + ijk[:, 1]) * dims[2] + ijk[:, 2]   # [N]

    # Sort points by key. Afterwards, points of the same cell are adjacent in memory.
    #   key_s : sorted keys
    #   order : order[i] = original index of the i-th sorted point (to map back later)
    # stable=True keeps ties in original index order => deterministic results.
    key_s, order = torch.sort(key, stable=True)

    # Compress runs of equal keys into: the list of OCCUPIED cells (ukeys, sorted)
    # and how many points each holds (counts). Empty cells cost nothing.
    ukeys, counts = torch.unique_consecutive(key_s, return_counts=True)

    # starts[c] = position in the sorted array where cell c's points begin.
    # (cumulative count minus own count = number of points in all earlier cells)
    starts = torch.cumsum(counts, 0) - counts

    # pts_sorted[start : start+count] are exactly the points of that cell.
    return dict(mn=mn, h=h, dims=dims, order=order, ukeys=ukeys,
                starts=starts, counts=counts, pts_sorted=pts[order])


def pick_cell_size(pts, target=8, iters=4):
    """
    Choose h automatically so an occupied cell holds ~`target` points on average.
    For points lying on surfaces, points-per-cell grows like h^2, hence the sqrt update.
    NOTE: if you also thin the reference set, you should FIX h yourself and thin at h/3
    (see thin()), otherwise cells can overflow M.
    """
    h = (pts.max(0).values - pts.min(0).values).max() / 200   # crude starting guess
    for _ in range(iters):
        c = build_grid(pts, h)['counts'].float().mean()       # mean points per occupied cell
        h = h * (target / c).sqrt()                           # rescale toward the target
    return float(h)


@torch.no_grad()
def grid_knn(g, q, K=12, M=32, chunk=8192, self_exclude=False):
    """
    For each query point, find its K nearest grid points that lie within radius h.

    Why only 27 cells? Cell side == search radius h, so any point within distance h of the
    query differs by < 1 cell per axis => it is in the query's own cell or an adjacent one.

    q:            [Q,3] query points
    self_exclude: only valid when q is the SAME array as the grid's reference points, in the
                  SAME order (used for the spacing self-query). Skips the point itself.
    Returns: out_d [Q,K] distances (inf where fewer than K neighbours exist within h),
             out_i [Q,K] indices into the ORIGINAL reference array (-1 where none).
    """
    dev, Q = q.device, q.shape[0]
    offs, ar = OFFS.to(dev), torch.arange(M, device=dev)
    out_d = torch.full((Q, K), float('inf'), device=dev)
    out_i = torch.full((Q, K), -1, dtype=torch.long, device=dev)
    dims, nu = g['dims'], g['ukeys'].numel()

    # Process queries in chunks to bound memory: the big temporary tensors are [n,27,M(,3)].
    for s in range(0, Q, chunk):
        qc = q[s:s + chunk]; n = qc.shape[0]

        # 1) Which 27 cells to look in: the query's own cell coordinate + each offset.
        nb = ((qc - g['mn']) / g['h']).floor().long()[:, None, :] + 1 + offs    # [n,27,3]

        # 2) Cells outside the grid's bounds contain nothing (also covers queries outside the grid).
        inside = ((nb >= 0) & (nb < dims)).all(-1)                              # [n,27]
        nbc = torch.minimum(nb.clamp(min=0), dims - 1)    # clamp only so key math stays valid

        # 3) Same flattening as in build_grid -> one key per (query, neighbour cell).
        key = (nbc[..., 0] * dims[1] + nbc[..., 1]) * dims[2] + nbc[..., 2]     # [n,27]

        # 4) Look the keys up in the sorted list of occupied cells (binary search).
        #    It is a HIT only if the cell is inside the grid AND actually occupied.
        pos = torch.searchsorted(g['ukeys'], key).clamp(max=nu - 1)             # [n,27]
        hit = inside & (g['ukeys'][pos] == key)

        # 5) For each hit cell: where its points start and how many (capped at M).
        #    CAP: points beyond the first M of a cell are IGNORED. Thinning guarantees this
        #    never triggers; without it, dense cells silently lose neighbours.
        start = torch.where(hit, g['starts'][pos], torch.zeros_like(pos))
        cnt = torch.where(hit, g['counts'][pos], torch.zeros_like(pos)).clamp(max=M)

        # 6) Build candidate slots [n,27,M]: slot j of a cell is valid if j < count.
        #    Invalid slots read index 0 (a harmless dummy) and are masked out via `ok`.
        ok = ar < cnt[..., None]                                                # [n,27,M]
        idx = torch.where(ok, start[..., None] + ar, torch.zeros_like(pos)[..., None])

        # 7) Squared distance from each query to every candidate; remember original ids.
        d2 = ((g['pts_sorted'][idx] - qc[:, None, None, :]) ** 2).sum(-1)       # [n,27,M]
        orig = g['order'][idx]                                                  # [n,27,M]

        # 8) Keep only candidates inside the search radius (and, optionally, not the query itself).
        ok &= d2 <= g['h'] ** 2
        if self_exclude:
            ok &= orig != (s + torch.arange(n, device=dev))[:, None, None]

        # 9) Flatten to [n, 27*M] and take the K smallest. Masked entries are inf and lose.
        d2 = torch.where(ok, d2, torch.full_like(d2, float('inf'))).view(n, -1)
        v, ti = d2.topk(K, dim=1, largest=False)              # v: K smallest d2, ti: their slots
        i = orig.view(n, -1).gather(1, ti)                    # map slots -> original indices
        out_d[s:s + n] = v.sqrt()
        out_i[s:s + n] = torch.where(torch.isfinite(v), i, torch.full_like(i, -1))
    return out_d, out_i


@torch.no_grad()
def reach_height(ref, qry, K=12, Kmin=6, M=32, target=8, eig_chunk=200_000, h=None):
    """
    ref: [N,3] trusted points (typically THINNED interior Gaussians)
    qry: [Q,3] points to score (e.g. class-1 Gaussians)
    Kmin: a query with fewer than Kmin trusted neighbours inside radius h is 'unsupported'
          (nothing trusted nearby -> reach/height are set to inf)

    Returns reach [Q], height [Q], unsupported [Q] bool, h (the cell size used).
    """
    h = pick_cell_size(ref, target) if h is None else h
    g = build_grid(ref, h)

    # --- A) Spacing field ---------------------------------------------------------------
    # For every reference point: mean distance to its K nearest OTHER reference points
    # (within h). This is "the typical gap between trusted Gaussians around here".
    d_ref, _ = grid_knn(g, ref, K=K, M=M, self_exclude=True)
    fin = torch.isfinite(d_ref)                                       # [N,K] valid neighbours
    sp = torch.where(fin, d_ref, torch.zeros_like(d_ref)).sum(1) / fin.sum(1).clamp(min=1)
    sp = torch.where(fin.any(1), sp, torch.full_like(sp, h))          # isolated ref point -> use h

    # --- B) Neighbours of each query ----------------------------------------------------
    d, j = grid_knn(g, qry, K=K, M=M)             # d: [Q,K] distances, j: [Q,K] ref indices (-1 = none)
    m = j >= 0                                    # [Q,K] which neighbour slots are real
    n = m.sum(1).clamp(min=1)                     # [Q]   number of real neighbours (>= 1 to avoid /0)
    mf = m.float()
    jj = j.clamp(min=0)                           # safe index for gathering; masked by mf afterwards

    # --- C) Local spacing at the query, then REACH --------------------------------------
    # Spacing at a query = average spacing of its own neighbours (uses the field from A).
    spacing = (sp[jj] * mf).sum(1) / n + 1e-8                                     # [Q]
    # REACH = mean distance to neighbours / local spacing.  ~1 => right next to trusted points.
    reach = (torch.where(m, d, torch.zeros_like(d)).sum(1) / n) / spacing         # [Q]

    # --- D) Plane fit by PCA over the neighbours' positions -----------------------------
    nbr = ref[jj]                                                  # [Q,K,3] neighbour positions
    c = (nbr * mf[..., None]).sum(1) / n[:, None]                  # [Q,3] centroid of real neighbours
    diff = (nbr - c[:, None]) * mf[..., None]                      # centred; masked slots become 0
    cov = torch.einsum('qki,qkj->qij', diff, diff) / n[:, None, None]   # [Q,3,3] covariance

    # Eigen-decomposition of the 3x3 covariances. Done on CPU in float64: batched eigh on the
    # GPU tried to allocate ~50 GiB. np.linalg.eigh returns eigenvalues in ASCENDING order, so
    # column 0 of the eigenvector matrix is the direction of LEAST spread = the surface NORMAL.
    # (The flat cloud is wide along the surface and thin across it.)
    # Note: one call on the whole array; no chunk loop needed (the loop in the earlier
    # adaptation recomputed everything on every iteration).
    cov64 = cov.double().cpu().numpy()                             # [Q,3,3]
    _, evec = np.linalg.eigh(cov64)                                # evec[..., :, 0] = smallest-eigenvalue vector
    normal = torch.from_numpy(evec[..., 0]).float().to(cov.device) # [Q,3]
    del cov64, evec

    # --- E) HEIGHT ------------------------------------------------------------------------
    # Distance from the query to the fitted plane = |(q - centroid) . normal|, in units of spacing.
    height = ((qry - c) * normal).sum(-1).abs() / spacing          # [Q]

    # --- F) Unsupported queries -----------------------------------------------------------
    unsupported = m.sum(1) < Kmin                  # too few trusted neighbours within radius h
    reach[unsupported] = float('inf'); height[unsupported] = float('inf')
    return reach, height, unsupported, h


def thin(pts, s):
    """
    Keep ONE point per small voxel of side s; returns the indices of the survivors.
    Purpose: bound the density so a grid cell of side h never holds more than M points.
    Rule: thin with s = h / 3  =>  a cell (h x h x h) contains at most 3^3 = 27 voxels
          => at most 27 points  <=  M = 32.
    CAVEAT: thinning puts a floor (about s) under the measured spacing, which biases any
    ratio that uses spacing in very dense regions.
    """
    mn = pts.min(0).values
    ijk = ((pts - mn) / s).floor().long()                   # voxel coordinate of each point
    dims = ijk.max(0).values + 1
    key = (ijk[:, 0] * dims[1] + ijk[:, 1]) * dims[2] + ijk[:, 2]
    key_s, order = torch.sort(key, stable=True)             # deterministic
    first = torch.ones_like(key_s, dtype=torch.bool)
    first[1:] = key_s[1:] != key_s[:-1]                     # True at the first point of each voxel
    return order[first]                                     # original indices of the survivors


# ----------------------------------------------------------------------------------------
# Typical usage (hold out ~10% of the trusted set to define "normal" scores):
#
#   h = 0.172                                   # fixed radius; thin at h/3 to match it
#   ref_t = ref_main[thin(ref_main, h / 3)]
#   reach, height, unsupported, _ = reach_height(ref_t, qry, K=12, Kmin=6, M=32, h=h)
#   # Sanity check: build_grid(ref_t, h)['counts'].max() should be <= 27.
# ----------------------------------------------------------------------------------------

In [ ]:
def p99(x): return float(np.percentile(x[torch.isfinite(x)].cpu().numpy(), 99))

In [ ]:
def generate_category_masks(N, top2_vals, sum_vals, entropies, coordinates, opacities, opacity_thres):
    NAMES  = ["unobserved", "interior", "ambiguous", "floaters", "default", "boundary"]
    lab = np.full(N, NAMES.index("default"), dtype=np.int8)

    top2_np = top2_vals.detach().cpu().numpy()
    S_np = sum_vals.detach().cpu().numpy()
    e_np = entropies.detach().cpu().numpy()
    opac_np = opacities.detach().cpu().numpy()
    xyz = coordinates.detach().cpu().numpy()

    observed = S_np > opt.seg_hist_min_evidence

    p1 = top2_np[:, 0]
    p2 = top2_np[:, 1]

    tail = 1.0 - p1 - p2  # residual prob mass outside top-2 (was S - p1 - p2: mass/prob unit bug)

    # Decided per 11-10-2026
    lab[observed & (p1 > TAU_P1) & (e_np < TAU_ENTROPY_LOW)] = NAMES.index("interior")
    lab[observed & (p1 <= TAU_P1) & (p2 > TAU_P2_HIGH) & (e_np > TAU_ENTROPY_LOW) & (e_np <= TAU_ENTROPY_HIGH) & (opac_np < opacity_thres)] = NAMES.index("floaters")
    lab[observed & (tail > TAU_TAIL_HIGH) & (e_np > TAU_ENTROPY_HIGH)] = NAMES.index("ambiguous")
    lab[observed & (tail > TAU_TAIL_HIGH) & (e_np > TAU_ENTROPY_HIGH) & (opac_np < opacity_thres)] = NAMES.index("floaters")

    lab[~observed] = NAMES.index("unobserved")

    # 3D-grid cells neighbouring using 3D kNN

    # Get ids of interior and default Gaussians
    # Interior Gaussians will be used as trusted neighbour
    # Default Gaussians will then marked as boundary, floaters, or as it is.
    ref_ids = np.flatnonzero(lab == NAMES.index("interior"))     # ids, not label values
    qry_ids = np.flatnonzero(lab == NAMES.index("default"))

    # Get interior & default Gaussians coordinates
    xyz_t = torch.from_numpy(xyz).float().cuda()
    ref = xyz_t[torch.from_numpy(ref_ids).cuda()].contiguous()
    qry = xyz_t[torch.from_numpy(qry_ids).cuda()].contiguous()

    # Cells are cube shaped with the dimension of h x h x h
    h = pick_cell_size(ref)

    # Hold-out subsample of Gaussians
    # and will be used to baseline off distance of every Gaussians
    # in the format of reach & height
    rng = np.random.default_rng(0)
    is_hold = rng.random(len(ref_ids)) < 0.10
    hold = ref[torch.from_numpy(is_hold).cuda()]
    ref_main = ref[torch.from_numpy(~is_hold).cuda()]

    ref_t = ref_main[thin(ref_main, h / 3)]

    reach_query, height_query, unsupported_query, _ = reach_height(ref_t, qry,  K=12, Kmin=6, M=32, h=h)
    reach_hold, height_hold, unsupported_hold, _ = reach_height(ref_t, hold, K=12, Kmin=6, M=32, h=h)

    thresh_reach, thresh_height = p99(reach_hold), p99(height_hold)
    supported_query = ~unsupported_query
    lo_reach, lo_height = reach_query <= thresh_reach, height_query <= thresh_height
    quad = {
        "supported": supported_query & lo_reach & lo_height,
        "inspect":  supported_query & lo_reach & ~lo_height,
        "defer":    supported_query & ~lo_reach & lo_height,
        "floater":         supported_query & ~lo_reach & ~lo_height,
        "unsupported":                       unsupported_query,
    }

    boundary_ids = qry_ids[quad['supported'].cpu().numpy()]
    lab[boundary_ids] = NAMES.index("boundary")

    if config['LVL2_FLOATERS']:
        floater_ids = qry_ids[quad['floater'].cpu().numpy()]
        lab[floater_ids] = NAMES.index("floaters")

    unobserved_mask = np.where(lab == NAMES.index("unobserved"), True, False)
    default_mask = np.where(lab == NAMES.index("default"), True, False)
    interior_mask = np.where(lab == NAMES.index("interior"), True, False)
    floaters_mask = np.where(lab == NAMES.index("floaters"), True, False)
    ambiguous_mask = np.where(lab == NAMES.index("ambiguous"), True, False)
    boundary_mask = np.where(lab == NAMES.index("boundary"), True, False)

    return unobserved_mask, default_mask, interior_mask, floaters_mask, ambiguous_mask, boundary_mask

### Train loop

In [ ]:
TAU_P1          = 0.85
TAU_P2_LOW      = 0.1
TAU_P2_HIGH     = 0.25
TAU_TAIL_LOW    = 0.1
TAU_TAIL_HIGH   = 0.25
TAU_ENTROPY_LOW = 0.25   # less than TAU_ENTROPY -> interior
TAU_ENTROPY_HIGH= 1.5

In [ ]:
from sgdense.densify_tracker import DensifyTracker
from sgdense.viz import LiveDensifier3D

# Instantiate the tracker bound to the live GaussianModel.
tr = DensifyTracker(gaussians, snapshot_max_points=5000)
tr.regime = "all"
print("Tracker initialised with", gaussians.get_xyz.shape[0], "Gaussians")

from sgdense.grad_stats import GradStatsLogger

# Accumulated-gradient diagnostics (the clone/split trigger + per-iteration grad).
grad_logger = GradStatsLogger(threshold=opt.densify_grad_threshold)
print("GradStatsLogger ready (densify_grad_threshold =", opt.densify_grad_threshold, ")")


> `seg_split_deterministic` (T0) was removed from the codebase; splits now use the
> vanilla random `N(0, Σ)` placement. This cell is intentionally blank.

In [ ]:
first_iter = 0
saving_iterations = [1000, 5000, 12000]
opt.seg_warmup_iters = 2000

seg_ce_window_sum = 0.0
photo_window_sum = 0.0
seg_photo_ratio = 0.0

writer = SummaryWriter(f"logs/{RUN_ID}")

gaussians.set_gaussian_properties_trainable(first_iter > opt.seg_warmup_iters)

# pbar = tqdm(range(first_iter, opt.iterations), desc="Training progress")
first_iter += 1

with tqdm(range(first_iter, opt.iterations+1), desc="Evaluating") as pbar:
  for iteration in pbar:
    iter_start.record()
    tr.step = iteration

    gaussians.update_learning_rate(iteration)

    if iteration == opt.seg_warmup_iters + 1:
        gaussians.set_gaussian_properties_trainable(True)

    # Every 1000 its we increase the levels of SH up to a maximum degree
    if iteration % 1000 == 0:
        gaussians.oneupSHdegree()

    if iteration < opt.seg_warmup_iters:
      for p in (gaussians._xyz, gaussians._features_dc, gaussians._features_rest,
              gaussians._opacity, gaussians._scaling, gaussians._rotation):
        p.requires_grad_(False)

    # Pick a random Camera
    if not viewpoint_stack:
        viewpoint_stack = scene.getTrainCameras().copy()
        viewpoint_indices = list(range(len(viewpoint_stack)))
    rand_idx = randint(0, len(viewpoint_indices) - 1)
    viewpoint_cam = viewpoint_stack.pop(rand_idx)
    vind = viewpoint_indices.pop(rand_idx)

    # Render
    # if (iteration - 1) == debug_from:
    #     pipe.debug = True

    bg = torch.rand((3), device="cuda") if opt.random_background else background

    render_pkg = render(viewpoint_cam, gaussians, pipe, bg, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE,num_segmentation_classes=dataset.num_segmentation_classes)
    image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

    if viewpoint_cam.alpha_mask is not None:
        alpha_mask = viewpoint_cam.alpha_mask.cuda()
        image *= alpha_mask

    if iteration == 1:
      # image_np = image.detach().cpu().numpy()
      # img_corrected = np.moveaxis(image_np, 0, -1)
      writer.add_image(f'Rendered sample {iteration}', image, iteration)

    if iteration % 1000 == 0:
      # image_np = image.detach().cpu().numpy()
      # img_corrected = np.moveaxis(image_np, 0, -1)
      writer.add_image(f'Rendered sample {iteration}', image, iteration)

    # Loss
    photo_weight = 0

    if iteration > opt.seg_warmup_iters:
      new_ph_weight = (iteration // opt.seg_warmup_iters) / 10
      photo_weight = new_ph_weight if new_ph_weight <= 1.0 else 1.0

    writer.add_scalar("L_photo weight", photo_weight, iteration)

    gt_image = viewpoint_cam.original_image.cuda()
    Ll1 = l1_loss(image, gt_image)

    if FUSED_SSIM_AVAILABLE:
        ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
    else:
        ssim_value = ssim(image, gt_image)

    loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim_value)
    photo_loss = loss.item()

    # Option A: photometric objective (weighted) BEFORE the segmentation term.
    # Its screen-space gradient feeds the densification trigger so L_seg cannot
    # inflate clone/split. Toggle via config['DENSIFY_PHOTO_GRAD_ONLY'].
    loss_no_seg = photo_weight * loss

    writer.add_scalar("L_photo", photo_loss, iteration)

    # Segmentation decode — AFTER rasterization, in the training loop (the
    # decoder compares decoded outputs across cameras for L_consist later).
    rendered_seg = render_pkg["rendered_seg"]                       # [NUM_SEG_CHANNELS, H, W]
    seg_logits = decoder(rendered_seg.permute(1, 2, 0))             # [H, W, num_segmentation_classes]
    gt_segmentation = getattr(viewpoint_cam, "gt_segmentation", None)
    Ll1seg = 0.0
    seg_ce_raw = None
    if gt_segmentation is not None:
        seg_ce_raw = torch.nn.functional.cross_entropy(
                seg_logits.reshape(-1, dataset.num_segmentation_classes),
                gt_segmentation.cuda().reshape(-1).long(),
                ignore_index=-1)  # void rule v1: ignore pixels labeled 255
        L_seg = opt.lambda_seg * seg_ce_raw
        loss = (photo_weight * loss) + L_seg
        Ll1seg = L_seg.item()

    writer.add_scalar("L_sem", Ll1seg, iteration)

    for param_group in decoder_optimizer.param_groups:
        param_group['lr'] = get_decoder_lr(iteration)

    # Option A: non-accumulating photometric-only screen-space gradient; the
    # full loss.backward() below still drives the optimiser unchanged.
    densify_point_grad = None
    if (config.get('DENSIFY_PHOTO_GRAD_ONLY', False) and loss_no_seg.requires_grad
            and gt_segmentation is not None
            and iteration < opt.densify_until_iter
            and iteration > opt.seg_warmup_iters):
        densify_point_grad = torch.autograd.grad(
            loss_no_seg, viewspace_point_tensor,
            retain_graph=True, allow_unused=True)[0]

    loss.backward()

    # Probe of freezing params
    if iteration == 100:
        assert gaussians._xyz.grad is None, "xyz not frozen"
        assert gaussians._segmentation_encoding.grad is not None, "encoding not training"

    iter_end.record()

    with torch.no_grad():
        # Raw per-iteration 2D-mean gradient norm (the gradient each Gaussian sees).
        # When decoupled, log the gradient that actually drives the trigger.
        densify_grad_for_log = (densify_point_grad if densify_point_grad is not None
                                else viewspace_point_tensor.grad)
        if densify_grad_for_log is not None:
            grad_logger.observe_iter(
                torch.norm(densify_grad_for_log[visibility_filter, :2], dim=-1),
                iteration)
        # Progress bar
        ema_loss_for_log = 0.4 * loss.item() + 0.6 * ema_loss_for_log
        ema_Ll1seg_for_log = 0.4 * Ll1seg + 0.6 * ema_Ll1seg_for_log

        if seg_ce_raw is not None:
            seg_ce_window_sum += seg_ce_raw.item()
            photo_window_sum += photo_loss
        if iteration % 100 == 0 and photo_window_sum > 0:
            seg_photo_ratio = seg_ce_window_sum / photo_window_sum
            # if tb_writer:
            writer.add_scalar("loss/seg_photo_ratio", seg_photo_ratio, iteration)
            seg_ce_window_sum = 0.0
            photo_window_sum = 0.0

        if iteration % 10 == 0:
          pbar.set_postfix({"Loss": f"{ema_loss_for_log:.{3}f}", "Seg Loss": f"{ema_Ll1seg_for_log:.{3}f}"})
          pbar.update(10)

        if (iteration in saving_iterations):
            print("\n[ITER {}] Saving Gaussians".format(iteration))
            scene.save(iteration)
            save_decoder_checkpoint(decoder, decoder_optimizer, iteration, scene.model_path + f"/decoder{iteration}.pth")

            np.savez(scene.model_path + f"/seg_readout_{iteration}.npz",
                      seg_hist=gaussians.seg_hist.detach().cpu().numpy())

        if iteration < opt.densify_until_iter and iteration > opt.seg_warmup_iters:
            # Keep track of max radii in image-space for pruning
            gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
            if densify_point_grad is not None:
                # Option A: accumulate only the photometric screen-space gradient
                # into the clone/split trigger (mirrors add_densification_stats).
                gaussians.xyz_gradient_accum[visibility_filter] += torch.norm(
                    densify_point_grad[visibility_filter, :2], dim=-1, keepdim=True)
                gaussians.denom[visibility_filter] += 1
            else:
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

            if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                _, _, _, entropy, top2, S = seg_hist_readout(gaussians.seg_hist)

                size_threshold = 20 if iteration > opt.opacity_reset_interval else None

                opacities = gaussians.get_opacity.squeeze(1)
                gauss_xyz = gaussians.get_xyz

                if config['LVL1_FLOATERS'] or config['LVL2_FLOATERS'] or config['POSITION_BASED_TRESH']:
                    unobserved_mask, default_mask, interior_mask, floaters_mask, ambiguous_mask, boundary_mask = generate_category_masks(opacities.shape[0], top2, S, entropy, gauss_xyz, opacities, 0.501)
                    other_mask = default_mask | ambiguous_mask
                    floaters_count = np.sum(floaters_mask)
                    writer.add_scalar("densify/floaters_flagged", float(floaters_count), iteration)

                else:
                    unobserved_mask = default_mask = interior_mask = floaters_mask = ambiguous_mask = boundary_mask = other_mask = None
                    floaters_count = None
                    # writer.add_text(f"floaters_{iteration}", f"{floaters_count}", iteration)

                floaters_mask_t = torch.tensor(floaters_mask, device='cuda')
                interior_mask_t = torch.tensor(interior_mask, device='cuda')
                boundary_mask_t = torch.tensor(boundary_mask, device='cuda')
                other_mask_t = torch.tensor(other_mask, device='cuda')

                if not config['LVL1_FLOATERS'] or not config['LVL2_FLOATERS']:
                    floaters_mask_t = None
                elif not config['POSITION_BASED_THRESH']:
                    interior_mask_t = None
                    boundary_mask_t = None
                    other_mask_t = None

                # Snapshot the accumulated trigger BEFORE densify_and_prune resets it.
                grad_logger.snapshot(gaussians.xyz_gradient_accum, gaussians.denom, iteration)

                densify_and_prune(gaussians, opt.densify_grad_threshold, 0.005, scene.cameras_extent,
                                            size_threshold, radii,
                                            interior_mask_t, boundary_mask_t, floaters_mask_t, other_mask_t)

            if iteration % opt.opacity_reset_interval == 0 or (dataset.white_background and iteration == opt.densify_from_iter):
                gaussians.reset_opacity()
                tr.on_reset()

            tr.flush(iteration)

        # Optimizer step
        if iteration < opt.iterations:
            gaussians.exposure_optimizer.step()
            gaussians.exposure_optimizer.zero_grad(set_to_none = True)
            if use_sparse_adam:
                visible = radii > 0
                gaussians.optimizer.step(visible, radii.shape[0])
                gaussians.optimizer.zero_grad(set_to_none = True)
            else:
                gaussians.optimizer.step()
                gaussians.optimizer.zero_grad(set_to_none = True)
            decoder_optimizer.step()
            decoder_optimizer.zero_grad(set_to_none = True)

writer.flush()
writer.close()

# Persist the densification event log for the post-hoc visualizer.
tr.save(os.path.join(str(OUT_DIR), "densify_events.pt"))
print(f"[tracker] {len(tr.events)} densification events logged -> densify_events.pt")
grad_logger.save(os.path.join(str(OUT_DIR), "grad_stats.json"))
print(f"[grad_stats] {len(grad_logger.events)} event snapshots, "
      f"{len(grad_logger.iters)} iter rows -> grad_stats.json")

# Quick look at the accumulated-gradient diagnostics
import json
import pandas as pd

gs = json.load(open(os.path.join(str(OUT_DIR), "grad_stats.json")))
ev = pd.DataFrame(gs["events"])
it = pd.DataFrame(gs["iters"], columns=["iter", "n_visible", "mean", "max"])
print("densify_grad_threshold:", gs["threshold"])
display(ev[["iter", "n", "mean", "p99", "frac_over", "thresh_pct"]].tail(10))
it.set_index("iter")["mean"].plot(title="per-iteration 2D-mean gradient norm", figsize=(11, 3))


In [ ]:
with open(os.path.join(OUT_DIR, "experiment_config.json"), "w") as f:
    json.dump(config, f, indent=1)

## Densification Visualization (post-hoc)

The `DensifyTracker` (from the `sgdense` package) was hooked into
`densify_and_prune` / `densify_and_clone` / `densify_and_split` during training.
It logs, for every densification event, which Gaussians met the **clone** /
**split** condition, their **children**, everything **pruned** (with reason),
and the population over time -- plus a stable id per Gaussian for lineage.

Reload the saved log without retraining:

```python
from sgdense import load_events
tr2 = load_events(os.path.join(str(OUT_DIR), "densify_events.pt"))
```


In [ ]:
from sgdense import load_events
tr2 = load_events(os.path.join(str(OUT_DIR), "densify_events.pt"))
tr = tr2

In [ ]:
import matplotlib.pyplot as plt
from sgdense.viz import (
    plot_timeline, plot_conditions, plot_prune_breakdown,
    plot_event_3d, show_event, show_lineage, replay, save_report,
)

# faint context cloud for the 3D panels
bg_xyz = gaussians.get_xyz.detach().cpu().numpy()

# thresholds actually used by densify_and_prune
grad_thr = float(opt.densify_grad_threshold)
size_thr = float(gaussians.percent_dense) * float(scene.cameras_extent)

if not len(tr.events):
    print("No densification events were logged.")
else:
    plot_timeline(tr); plt.show()
    plot_conditions(tr, grad_thr=grad_thr, size_thr=size_thr); plt.show()
    plot_prune_breakdown(tr); plt.show()

    mid = tr.events[len(tr.events) // 2]["iter"]
    show_event(tr, mid, background_xyz=bg_xyz); plt.show()

    # pick a Gaussian that was densified and trace its lineage
    e0 = tr.events[0]
    if e0["split"]:
        seed = int(e0["split"][0]["parent_gid"][0])
    elif e0["clone"]:
        seed = int(e0["clone"][0]["parent_gid"][0])
    elif e0["prune"]:
        seed = int(e0["prune"][0]["gid"][0])
    else:
        seed = None
    if seed is not None:
        show_lineage(tr, seed)

# Offline replay of the whole process as a 3D animation (works from a reloaded
# log too). Uncomment to re-watch it:
# replay(tr, background_xyz=bg_xyz, delay=0.02)

# One-file HTML report: interactive 3D replay + timeline / conditions /
# prune panels + the experiment config + a per-event summary table.
save_report(tr, str(OUT_DIR / "densification_report.html"),
            config=str(OUT_DIR / "experiment_config.json"),
            background_xyz=bg_xyz, grad_thr=grad_thr, size_thr=size_thr)


## Evaluation

In [ ]:
!pip install lpips torchmetrics

In [ ]:
RUN_ID = 'run40'
OUT_DIR = HOME_DIR / 'output' / RUN_ID

In [ ]:
# ===== On-the-fly photometric + semantic eval (no images saved) =====
import sys, os, numpy as np, torch
import matplotlib.pyplot as plt                                   # NEW
REPO        = "/content/gaussian-splatting"
MODEL_PATH  = str(OUT_DIR)
ITERATION   = 12000
NUM_CLASSES = 100
IGNORE_ID   = 255
DEVICE      = "cuda"
NUM_SAMPLES = 4                                                   # NEW: views to plot per split

sys.path.insert(0, REPO)
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, get_combined_args
from scene import Scene, GaussianModel
from gaussian_renderer import render
import lpips
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
from pathlib import PosixPath
from scene.segmentation_decoder import SegmentationDecoder, load_decoder_checkpoint
import arguments

import builtins

builtins.np = np
builtins.numpy = np


arguments.PosixPath = PosixPath

sys.argv = ["eval", "-m", MODEL_PATH, "--iteration", str(ITERATION)]
parser = ArgumentParser()
lp_args, pp_args = ModelParams(parser, sentinel=True), PipelineParams(parser)
parser.add_argument("--iteration", default=-1, type=int)
args = get_combined_args(parser)

# decoder = SegmentationDecoder(dataset.seg_encoding_dim, dataset.num_segmentation_classes).cuda()
# decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=opt.decoder_lr_init)

# load_decoder_checkpoint(decoder, decoder_optimizer, OUT_DIR / f"decoder{ITERATION}.pth")

dataset, pipe = lp_args.extract(args), pp_args.extract(args)
gaussians = GaussianModel(dataset.sh_degree)
scene = Scene(dataset, gaussians, load_iteration=args.iteration, shuffle=False)
bg = torch.tensor([1, 1, 1] if dataset.white_background else [0, 0, 0],
                  dtype=torch.float32, device=DEVICE)

splits = {"train": scene.getTrainCameras(), "test": scene.getTestCameras()}
splits = {k: v for k, v in splits.items() if len(v) > 0}
print({k: len(v) for k, v in splits.items()})

# --- semantic hook ---
def render_semantic_ids(view):
    pkg = render(view, gaussians, pipe, bg)
    seg_feat = pkg["rendered_seg"]                                # [32,H,W]
    logits = decoder(seg_feat[None].permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
    return logits.argmax(1)[0]

def get_gt_semantic(view):
    return view.gt_segmentation.to(DEVICE).long().squeeze()       # squeeze handles [1,H,W] -> [H,W]

# --- NEW: visualization helpers ---
_rng = np.random.RandomState(0)
PALETTE = _rng.randint(40, 255, size=(NUM_CLASSES, 3)).astype(np.uint8)   # fixed colors per class

def colorize(ids):
    """[H,W] long tensor -> [H,W,3] uint8. Ignore/out-of-range pixels are black."""
    ids = ids.cpu().numpy()
    out = np.zeros((*ids.shape, 3), dtype=np.uint8)
    valid = (ids != IGNORE_ID) & (ids >= 0) & (ids < NUM_CLASSES)
    out[valid] = PALETTE[ids[valid]]
    return out

def to_img(t):
    """[3,H,W] float tensor -> [H,W,3] float numpy in [0,1]."""
    return t.clamp(0, 1).permute(1, 2, 0).cpu().numpy()

def plot_samples(samples, name, col_width=4.0, show_names=True):
    """Compact grid: one row per view, columns = GT RGB | Rendered RGB | GT Seg | Rendered Seg."""
    cols = ["GT RGB", "Rendered RGB", "GT Segmentation", "Rendered Segmentation"]
    n = len(samples)

    # size the figure from the real image aspect ratio so cells have no leftover whitespace
    _, H, W = samples[0]["gt_rgb"].shape
    row_height = col_width * H / W
    fig, axes = plt.subplots(
        n, 4,
        figsize=(col_width * 4, row_height * n + 0.5),            # +0.5 for the titles
        squeeze=False,
        gridspec_kw=dict(wspace=0.01, hspace=0.01),
        dpi=300
    )

    for i, s in enumerate(samples):
        imgs = [to_img(s["gt_rgb"]), to_img(s["pred_rgb"]),
                colorize(s["gt_seg"]), colorize(s["pred_seg"])]
        for j, im in enumerate(imgs):
            ax = axes[i, j]
            ax.imshow(im, aspect="auto")
            ax.axis("off")
            if i == 0:
                ax.set_title(cols[j], fontsize=12, pad=4)
        if show_names:
            axes[i, 0].text(0.02, 0.97, f"{s['name']} | PSNR {s['psnr']:.2f}",
                            transform=axes[i, 0].transAxes, ha="left", va="top",
                            fontsize=8, color="white",
                            bbox=dict(facecolor="black", alpha=0.55, pad=2, edgecolor="none"))

    fig.suptitle(f"[{name}] sample views", fontsize=13, y=1.0)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1 - 0.5 / (row_height * n + 0.5),
                        wspace=0.01, hspace=0.01)
    plt.show()

# --- evaluation loop ---
lp = lpips.LPIPS(net="vgg").to(DEVICE).eval()
DO_SEMANTIC = False

@torch.no_grad()
def evaluate(cams, name):
    psnrs, ssims, lpipss = [], [], []
    conf = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=DEVICE)
    per_view = []
    samples = []                                                  # NEW
    sample_idx = set(np.linspace(0, len(cams) - 1, min(NUM_SAMPLES, len(cams)), dtype=int).tolist())  # NEW

    for i, v in enumerate(cams):
        r = render(v, gaussians, pipe, bg)["render"].clamp(0, 1)[None]
        g = v.original_image[0:3].to(DEVICE)[None]
        mse = ((r - g) ** 2).mean().item()
        p = -10 * np.log10(max(mse, 1e-12))
        psnrs.append(p)
        ssims.append(ssim_fn(r, g, data_range=1.0).item())
        lpipss.append(lp(r * 2 - 1, g * 2 - 1).item())
        per_view.append((v.image_name, p))

        pred = gt = None
        if DO_SEMANTIC:
            pred, gt = render_semantic_ids(v), get_gt_semantic(v)
            m = (gt != IGNORE_ID) & (gt < NUM_CLASSES)
            conf += torch.bincount(gt[m] * NUM_CLASSES + pred[m].clamp(0, NUM_CLASSES - 1),
                                   minlength=NUM_CLASSES**2).view(NUM_CLASSES, NUM_CLASSES)

        if i in sample_idx:                                       # NEW: keep a few views for plotting
            samples.append(dict(
                name=v.image_name, psnr=p,
                gt_rgb=g[0].cpu(), pred_rgb=r[0].cpu(),
                gt_seg=gt.cpu() if gt is not None else None,
                pred_seg=pred.cpu() if pred is not None else None,
            ))

    print(f"\n[{name}] {len(cams)} views | PSNR {np.mean(psnrs):.2f} | SSIM {np.mean(ssims):.4f} | LPIPS {np.mean(lpipss):.4f}")
    print("  worst 3 by PSNR:", [(n, round(p, 2)) for n, p in sorted(per_view, key=lambda x: x[1])[:3]])

    if DO_SEMANTIC:
        c = conf.double().cpu().numpy()
        tp, gt_cnt, pr_cnt = np.diag(c), c.sum(1), c.sum(0)
        union = gt_cnt + pr_cnt - tp
        present = gt_cnt > 0
        iou = tp / np.maximum(union, 1)
        freq = gt_cnt / gt_cnt.sum()
        print(f"  semantic: pixel acc {tp.sum()/c.sum():.4f} | mIoU {iou[present].mean():.4f} | "
              f"fwIoU {(freq[present]*iou[present]).sum():.4f}")
        worst = np.argsort(np.where(present, iou, np.inf))[:5]
        print("  lowest-IoU classes:", [(int(k), round(float(iou[k]), 3)) for k in worst])
        plot_samples(samples, name)                               # NEW
    return samples

for name, cams in splits.items():
    evaluate(cams, name)

In [ ]:
gaussians.get_xyz.shape[0]

In [ ]:
# Colour every Gaussian by its SGDense category and export the FULL model
# (all Gaussians + scale/rotation/opacity/SH) to a .ply, colour = category.
# Run after training (or on a restored model) once seg_hist is populated.
import os
import numpy as np
import torch
from plyfile import PlyData, PlyElement

with np.load(f"{OUT_DIR}/seg_readout_{ITERATION}.npz") as loaded_seg_hist:
  seg_hist = {key: loaded_seg_hist[key] for key in loaded_seg_hist.files}

seg_hist_tensor = torch.from_numpy(seg_hist['seg_hist']).cuda()
del seg_hist

import gc
gc.collect()

_, _, _, entropy, top2, S = seg_hist_readout(seg_hist_tensor)

opacities = gaussians.get_opacity.squeeze(1)
xyz = gaussians.get_xyz

(unobserved_mask, default_mask, interior_mask, floaters_mask,
 ambiguous_mask, boundary_mask) = generate_category_masks(
    opacities.shape[0], top2, S, entropy, xyz, opacities, 0.501)

# category order matches the mask-return tuple
CATS = ["unobserved", "default", "interior", "floaters", "ambiguous", "boundary"]
MASKS = {"unobserved": unobserved_mask, "default": default_mask,
         "interior": interior_mask, "floaters": floaters_mask,
         "ambiguous": ambiguous_mask, "boundary": boundary_mask}
COLORS = {"unobserved": (0.62, 0.62, 0.62), "default": (0.23, 0.23, 0.23),
          "interior": (0.17, 0.63, 0.17), "floaters": (0.84, 0.15, 0.16),
          "ambiguous": (1.00, 0.50, 0.05), "boundary": (0.12, 0.47, 0.71)}

# disjoint partition -> exactly one True mask per Gaussian
M = np.stack([MASKS[c] for c in CATS], axis=1)          # [N, 6]
label = M.argmax(axis=1)                                # [N]
n = len(label)

print(f"{n:,} Gaussians")
for i, c in enumerate(CATS):
    k = int(M[:, i].sum())
    print(f"  {c:<11} {k:>9,}  ({k / n:5.2%})")

# per-Gaussian category RGB -> SH DC (rgb = C0*f_dc + 0.5) so any 3DGS
# viewer renders the category colour.
rgb = np.array([COLORS[CATS[i]] for i in label], dtype=np.float32)   # [N, 3]
C0 = 0.28209479177387814
f_dc = (rgb - 0.5) / C0                                              # [N, 3]

FORCE_OPAQUE = False   # True -> save every Gaussian fully opaque (vivid colour)

# Minimal attribute set: xyz, colour(f_dc), opacity, scale, rotation
# (raw stored values, same convention as the standard 3DGS PLY).
xyz_np = xyz.detach().cpu().numpy().astype(np.float32)
opacity = gaussians._opacity.detach().cpu().numpy().astype(np.float32)
if FORCE_OPAQUE:
    opacity = gaussians.inverse_opacity_activation(
        torch.ones_like(gaussians._opacity)).detach().cpu().numpy().astype(np.float32)
scale = gaussians._scaling.detach().cpu().numpy().astype(np.float32)
rot = gaussians._rotation.detach().cpu().numpy().astype(np.float32)

fields = ["x", "y", "z", "f_dc_0", "f_dc_1", "f_dc_2",
          "opacity", "scale_0", "scale_1", "scale_2",
          "rot_0", "rot_1", "rot_2", "rot_3"]
attrs = np.concatenate([xyz_np, f_dc, opacity, scale, rot], axis=1).astype(np.float32)

elements = np.empty(n, dtype=[(f, 'f4') for f in fields])
elements[:] = list(map(tuple, attrs))
out_ply = os.path.join(str(OUT_DIR), "category_colored.ply")
PlyData([PlyElement.describe(elements, 'vertex')]).write(out_ply)

print("saved:", out_ply, f"({n:,} Gaussians: xyz, f_dc, opacity, scale, rot)")